# Diagnose where real photos score

On DiffusionFace every detector ranks fakes above reals only weakly (AUC 0.69 to 0.76), and at the threshold that catches 95% of the fakes it flags 77% to 90% of the real photos (`docs/results.md`). Two causes would give the same AUC:

- **Low resolution.** DiffusionFace images are 256 x 256 CelebA-HQ portraits, so a face is about half the image wide and `square_crop` enlarges it. The detectors may read the missing detail as a sign of generation.
- **The real photos themselves.** CelebA-HQ was made from CelebA by a JPEG-artifact-removal network and a 4x super-resolution network (Karras et al., ICLR 2018, appendix C), so its "real" photos already went through generative networks.

This notebook scores real crops the detectors were not trained on (lab test side, FaceForensics++ test side, Celeb-DF, DiffusionFace) at their native resolution and lowered to faces `SIDES` px wide with `lower_resolution`. If lowering the resolution of lab and FaceForensics++ reals pushes their scores to where DiffusionFace reals sit, resolution explains the failure; if not, the CelebA-HQ processing does. Each crop's sharpness (variance of the Laplacian) shows which lowered resolution matches DiffusionFace.

The threshold of every detector is the 5th percentile of its DiffusionFace fake scores, the APCER 5% point of the DiffusionFace report. The result goes to `HF_MODEL/diagnosis/bona_fide_scores.json`.

**Runtime:** Colab Pro, L4 or A100, about 20 minutes. **Needs** the Colab secret `HF_TOKEN` and access to the private datasets.

### Configuration

In [ ]:
REPO_SOURCE = "git+https://github.com/paulohenriquevn/face-decision-lab@workspace"
HF_DATASET = "paulohenriquevn/face-decision-data"
FFPP_DATASET = "paulohenriquevn/face-decision-ffpp"
CELEBDF_DATASET = "paulohenriquevn/face-decision-celebdf-test"
DIFFUSIONFACE_DATASET = "paulohenriquevn/face-decision-diffusionface-test"
HF_MODEL = "paulohenriquevn/face-decision-clip-ln"
REPORT_PATH = "diagnosis/bona_fide_scores.json"
CLIP_MODEL = "openai/clip-vit-large-patch14"
GEND_MODELS = {"clip": "yermandy/GenD_CLIP_L_14", "pe": "yermandy/GenD_PE_L"}
GEND_FACE_SCALE = 1.3
V1_CHECKPOINT = "subject_holdout/trained_parameters.pt"
V2_CHECKPOINT = "openai-ce-e1-b32-lr0.0003-fs1.5-g10-noaug/lab_ffpp_real/checkpoint.pt"
SIDES = (128, 96, 64)  # face widths, in px, that each real crop is lowered to
PER_SOURCE = 1000  # real crops scored per source dataset

In [ ]:
%%capture
!pip install -q "face-decision-lab @ {REPO_SOURCE}" "timm>=1.0.20"

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token = userdata.get("HF_TOKEN"))

### Real crops

The lab and FaceForensics++ reals come from the test side of `subject_holdout` and `ffpp`, which v1, v2 and GenD never trained on. Celeb-DF and DiffusionFace are test-only.

In [ ]:
import random
from face_decision_lab.hub import load_hub_samples
from face_decision_lab.protocol import protocol_split

images, real_by_source = {}, {}

def load(repo, prefix):
    samples, hub_images = load_hub_samples(repo, key_prefix = prefix)
    images.update(zip([s.image_path for s in samples], hub_images, strict = True))
    return samples

lab, ffpp = load(HF_DATASET, "hub"), load(FFPP_DATASET, "faceforensics")
celebdf, diffusionface = load(CELEBDF_DATASET, "celebdf"), load(DIFFUSIONFACE_DATASET, "diffusionface")
reals = [s for s in protocol_split(lab, "subject_holdout")[2] + protocol_split(lab + ffpp, "ffpp")[2] + celebdf + diffusionface
         if s.is_bona_fide]
for source in sorted({s.source_dataset for s in reals}):
    group = [s for s in reals if s.source_dataset == source]
    real_by_source[source] = random.Random(0).sample(group, min(PER_SOURCE, len(group)))
diffusionface_fakes = [s for s in diffusionface if not s.is_bona_fide]
print({k: len(v) for k, v in real_by_source.items()}, len(diffusionface_fakes), "DiffusionFace fakes")

### Detectors

The same loaders as `score_diffusion_benchmarks.ipynb`. Every crop is lowered before GenD's tighter framing is cut from it, as a small face in the source would be.

In [ ]:
import importlib.util, io, json
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import requests
import torch
from huggingface_hub import HfApi, hf_hub_download
from PIL import Image
from safetensors.torch import load_file
from torch import nn
from tqdm.auto import tqdm
from transformers import CLIPImageProcessor, CLIPVisionModel, CLIPVisionModelWithProjection
from face_decision_lab.faces import lower_resolution, tighten_crop
from face_decision_lab.rows import open_image

api = HfApi()
pool = ThreadPoolExecutor(max_workers = 8)
EXAMPLES = ["https://github.com/yermandy/deepfake-detection/blob/main/datasets/FF/DF/000_003/000.png?raw=true",
            "https://github.com/yermandy/deepfake-detection/blob/main/datasets/FF/real/000/000.png?raw=true"]
example_frames = [Image.open(io.BytesIO(requests.get(u, timeout = 30).content)).convert("RGB") for u in EXAMPLES]

def load_gend(repo):
    spec = importlib.util.spec_from_file_location(f"modeling_{repo.split('/')[1]}", hf_hub_download(repo, "modeling_gend.py"))
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    model = module.GenD(module.GenDConfig.from_pretrained(repo))
    loaded = model.load_state_dict(load_file(hf_hub_download(repo, "model.safetensors")), strict = False)
    assert not loaded.missing_keys and not loaded.unexpected_keys, loaded
    model = model.cuda().eval()
    with torch.no_grad():
        fake = model(torch.stack([model.feature_extractor.preprocess(f) for f in example_frames]).cuda()).softmax(-1)[:, 1].tolist()
    assert fake[0] > 0.5 > fake[1], (repo, fake)
    return model

processor = CLIPImageProcessor.from_pretrained(CLIP_MODEL)

class LayerNormTunedCLIPv1(nn.Module):  # the v1 detector, as trained at commit b66500a
    def __init__(self):
        super().__init__()
        self.encoder = CLIPVisionModelWithProjection.from_pretrained(CLIP_MODEL)
        self.head = nn.Linear(self.encoder.config.projection_dim, 1)

    def forward(self, pixels):
        return torch.sigmoid(self.head(nn.functional.normalize(self.encoder(pixel_values = pixels).image_embeds, dim = -1)).squeeze(-1))

class LayerNormTunedCLIP(nn.Module):  # clip_layernorm_detector.ipynb's detector (pooled features, two-class head)
    def __init__(self):
        super().__init__()
        self.encoder = CLIPVisionModel.from_pretrained(CLIP_MODEL)
        self.head = nn.Linear(self.encoder.config.hidden_size, 2)

    def forward(self, pixels):
        return self.head(nn.functional.normalize(self.encoder(pixel_values = pixels).pooler_output, dim = -1)).softmax(-1)[:, 1]

def load_v1():
    model = LayerNormTunedCLIPv1()
    model.load_state_dict(torch.load(hf_hub_download(HF_MODEL, V1_CHECKPOINT)), strict = False)
    return model.cuda().eval()

def load_v2():
    model = LayerNormTunedCLIP()
    loaded = model.load_state_dict(torch.load(hf_hub_download(HF_MODEL, V2_CHECKPOINT), weights_only = False)["trainable"], strict = False)
    assert not loaded.unexpected_keys, loaded.unexpected_keys[:5]
    return model.cuda().eval()

In [ ]:
import cv2

def crop_of(sample, side):
    image = open_image(images[sample.image_path])
    return image if side is None else lower_resolution(image, side)

def sharpness(sample, side):
    return float(cv2.Laplacian(cv2.cvtColor(np.asarray(crop_of(sample, side)), cv2.COLOR_RGB2GRAY), cv2.CV_64F).var())

def scores_of(part, prepare, model, side):
    out = []
    with torch.no_grad(), torch.autocast("cuda", dtype = torch.bfloat16):
        for i in tqdm(range(0, len(part), 128), leave = False):
            batch = list(pool.map(lambda s: prepare(s, side), part[i:i + 128]))
            out += model(torch.stack(batch).cuda()).float().cpu().tolist()
    return np.array(out)

def clip_pixels(sample, side):  # the lab detectors trained on the 1.5x crops as built
    return processor(images = crop_of(sample, side), return_tensors = "pt")["pixel_values"][0]

CONDITIONS = (None,) + SIDES

def score_all(prepare, model):
    """Scores of the DiffusionFace fakes, and of every source's reals under every condition."""
    reals = {(source, side): scores_of(part, prepare, model, side)
             for source, part in real_by_source.items() for side in CONDITIONS}
    return scores_of(diffusionface_fakes, prepare, model, None), reals

results = {}
for name, repo in GEND_MODELS.items():
    model = load_gend(repo)

    def gend_pixels(sample, side, m = model):
        return m.feature_extractor.preprocess(tighten_crop(crop_of(sample, side), GEND_FACE_SCALE))

    results[f"gend_{name}"] = score_all(gend_pixels, lambda x, m = model: m(x).softmax(-1)[:, 1])
    del model
results["gend_ensemble_clip_pe"] = ((results["gend_clip"][0] + results["gend_pe"][0]) / 2,
                                    {k: (results["gend_clip"][1][k] + results["gend_pe"][1][k]) / 2 for k in results["gend_clip"][1]})
results["layernorm_clip_v1"] = score_all(clip_pixels, load_v1())
results["v2_ffpp_real"] = score_all(clip_pixels, load_v2())
sharpness_by = {(source, side): float(np.median(list(pool.map(lambda s, side = side: sharpness(s, side), part))))
                for source, part in real_by_source.items() for side in CONDITIONS}

### Report

Per detector, source and resolution: the mean real score, its quartiles, and the share of reals above the DiffusionFace APCER 5% threshold (the BPCER they would add at that point). Sharpness is the median variance of the Laplacian of each condition's crops.

In [ ]:
def label(side):
    return "native" if side is None else f"{side}px"

report = {"sharpness": {f"{source}/{label(side)}": v for (source, side), v in sharpness_by.items()}, "detectors": {}}
for detector, (fakes, reals) in results.items():
    threshold = float(np.quantile(fakes, 0.05))
    rows = {f"{source}/{label(side)}": {"mean": float(v.mean()), "quartiles": np.quantile(v, [0.25, 0.5, 0.75]).tolist(),
                                       "above_threshold": float((v > threshold).mean())}
            for (source, side), v in reals.items()}
    report["detectors"][detector] = {"threshold": threshold, "fake_mean": float(fakes.mean()), "reals": rows}
    print(f"\n{detector}: threshold {threshold:.3f}, fake mean {fakes.mean():.3f}")
    for key, row in rows.items():
        print(f"  {key:34s} mean {row['mean']:.3f}  above {row['above_threshold']:.3f}  sharpness {report['sharpness'][key]:9.1f}")
api.upload_file(path_or_fileobj = json.dumps(report, indent = 2).encode(), repo_id = HF_MODEL, path_in_repo = REPORT_PATH,
                commit_message = "Real-photo scores by source and resolution")